# 4.4 · 岭回归 / Ridge Regression (L2)

> **课程定位 / Where this fits**
> **Part 4 第 4 课**。4.3 的过拟合和 4.2 的多重共线都有一个共同解药：**L2 正则化**。Ridge = 线性回归 + 对大系数的惩罚。2.10 节我们已证明 **Ridge = 高斯先验的 MAP**——这一课把它落到实处。
> Both 4.3's overfitting and 4.2's multicollinearity share one cure: L2 regularization. We proved Ridge = Gaussian-prior MAP in 2.10; now we apply it.

> 💡 **面试相关 / Interview-relevant**
> - "Ridge 和普通线性回归区别" ★★★★★
> - "L2 正则化为什么能防过拟合" ★★★★
> - "Ridge 为什么能解多重共线" ★★★★（XᵀX+λI 总可逆）
> - "λ 怎么选" ★★★★（CV）

---

## 学习目标 / Learning Objectives
1. 写出 Ridge 目标函数与**闭式解**, 看清 $+\lambda I$ 的作用。
2. 理解 Ridge **解多重共线**的数学原理（让 $\mathbf{X}^\top\mathbf{X}$ 必可逆）。
3. 看 **系数收缩路径**（λ 增大系数如何向 0 收缩但不为 0）。
4. 用 CV 选 λ, 理解偏差-方差的另一种调节方式。
5. 重申 **Ridge = 高斯先验 MAP**（2.10 接口）。

## 目录 / TOC
1. [目标函数 + 闭式解 ⭐](#1)
2. [数据](#2)
3. [从零实现 + 对照 sklearn](#3)
4. [Ridge 治多重共线 ⭐](#4)
5. [系数收缩路径](#5)
6. [CV 选 λ + 偏差方差](#6)
7. [= 高斯先验 MAP (2.10)](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 目标函数 + 闭式解 ⭐ / Objective & Closed Form

**普通 OLS**：$\min \|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2$
**Ridge**：加一个 **L2 惩罚项**, 压制大系数：
$$J(\mathbf{w}) = \|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2 + \lambda \|\mathbf{w}\|_2^2$$

求梯度置 0（和 4.1 一样推, 多一项 $2\lambda\mathbf{w}$）：
$$-2\mathbf{X}^\top(\mathbf{y}-\mathbf{X}\mathbf{w}) + 2\lambda\mathbf{w} = 0 \;\Rightarrow\; \boxed{\hat{\mathbf{w}} = (\mathbf{X}^\top\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^\top\mathbf{y}}$$

**关键洞察**：和 4.1 正规方程只差一个 **$+\lambda\mathbf{I}$**：
- $\lambda=0$：退化为 OLS
- $\lambda\to\infty$：系数全压向 0（极端高偏差）
- $+\lambda\mathbf{I}$ 让矩阵**对角占优 → 总是可逆**（即使 $\mathbf{X}^\top\mathbf{X}$ 奇异）→ 这就是治共线的数学。

⚠ **偏置 $w_0$ 不该被惩罚**（否则强行把预测拉向 0）, 且**必须先标准化**（否则惩罚对不同量纲不公, 3.4 节）。
The bias term should not be penalized, and features must be standardized first.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
X = data.data.values; y = data.target.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)
scaler = StandardScaler().fit(X_tr)
Xtr, Xte = scaler.transform(X_tr), scaler.transform(X_te)
print(f"{X.shape}, 已标准化 (Ridge 必须)")


<a id="3"></a>
## 3. 从零实现 + 对照 sklearn / From Scratch


In [ ]:
def fit_ridge(X, y, lam):
    # 不惩罚偏置: 单独处理截距 / don't penalize intercept
    n, d = X.shape
    Xb = np.c_[np.ones(n), X]                      # 加偏置列
    I = np.eye(d+1); I[0,0] = 0                     # 偏置项不惩罚 (对角第0个置0)
    w = np.linalg.solve(Xb.T @ Xb + lam*I, Xb.T @ y)
    return w[0], w[1:]                              # intercept, coefs

intercept, coefs = fit_ridge(Xtr, y_tr, lam=10.0)

from sklearn.linear_model import Ridge
ridge = Ridge(alpha=10.0).fit(Xtr, y_tr)
print(f"从零 vs sklearn 系数最大差异: {np.abs(coefs - ridge.coef_).max():.6f}")
print(f"截距: 从零={intercept:.4f}, sklearn={ridge.intercept_:.4f}")
print("\n💡 sklearn 的 alpha 就是公式里的 λ")


<a id="4"></a>
## 4. Ridge 治多重共线 ⭐ / Ridge Fixes Multicollinearity

4.2 节的 VIF 警告：共线时 OLS 系数**不稳定**（数据微变系数大跳, 甚至符号反）。Ridge 的 $+\lambda\mathbf{I}$ **稳定了解**。下面人为造强共线特征对比：


In [ ]:
# 造两个几乎相同的特征 (完美共线) / two nearly-identical features
n = 200
x1 = rng.normal(0, 1, n)
x2 = x1 + rng.normal(0, 0.01, n)        # x2 ≈ x1 (强共线)
y_syn = 3*x1 + rng.normal(0, 0.5, n)    # 真实只依赖 x1, 系数 3
Xc = np.c_[x1, x2]

from sklearn.linear_model import LinearRegression
print("真实关系: y = 3·x1 (x2 是 x1 的近似副本)\n")
# OLS: 系数在两个共线特征间任意分配, 不稳定 / OLS coefs unstable
ols = LinearRegression().fit(Xc, y_syn)
print(f"OLS:   w1={ols.coef_[0]:+.2f}, w2={ols.coef_[1]:+.2f}  (和≈3 但各自乱分, 可能一正一负)")
# Ridge: 把系数平摊到两个共线特征, 稳定 / Ridge spreads coefs, stable
rdg = Ridge(alpha=1.0).fit(Xc, y_syn)
print(f"Ridge: w1={rdg.coef_[0]:+.2f}, w2={rdg.coef_[1]:+.2f}  (平摊到两者, 稳定且合理)")
print("\nRidge 倾向于把系数'平摊'给共线特征, 避免 OLS 的极端不稳定")


<a id="5"></a>
## 5. 系数收缩路径 / Coefficient Shrinkage Path

**Ridge 的标志性图**：λ 从小到大, 看每个系数如何**平滑收缩向 0**（但永不精确等于 0——这是和 Lasso 4.5 的关键区别）。


In [ ]:
alphas = np.logspace(-2, 4, 50)
coef_path = np.array([Ridge(alpha=a).fit(Xtr, y_tr).coef_ for a in alphas])

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, name in enumerate(data.feature_names):
    ax.plot(alphas, coef_path[:, i], label=name)
ax.set_xscale("log"); ax.set_xlabel("λ (alpha)"); ax.set_ylabel("系数")
ax.axhline(0, color="k", lw=0.5)
ax.legend(fontsize=7, ncol=2); ax.set_title("Ridge 系数收缩路径: λ↑ 所有系数平滑趋0 (但不为0)")
plt.tight_layout(); plt.show()
print("所有系数随 λ 增大平滑收缩, 但都不会精确变 0 → Ridge 不做特征选择")
print("(Lasso 4.5 会让部分系数精确变 0 → 自动特征选择, 这是核心区别)")


<a id="6"></a>
## 6. CV 选 λ + 偏差方差 / Selecting λ via CV

λ 是**偏差-方差的旋钮**（4.3 是阶数, 这里是 λ）：
- λ 小 → 接近 OLS → 低偏差高方差（可能过拟合）
- λ 大 → 强收缩 → 高偏差低方差（可能欠拟合）

用 `RidgeCV`（内置高效 LOOCV）选最优 λ。


In [ ]:
from sklearn.linear_model import RidgeCV

alphas = np.logspace(-3, 3, 50)
# 手动 CV 画曲线 / manual CV curve
cv_scores = [cross_val_score(Ridge(alpha=a), Xtr, y_tr, cv=5, scoring="r2").mean() for a in alphas]
best_alpha = alphas[np.argmax(cv_scores)]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, cv_scores, "o-")
ax.axvline(best_alpha, color="r", ls="--", label=f"最优 λ={best_alpha:.3f}")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("CV R²"); ax.legend()
ax.set_title("CV 选 λ: 中等 λ 最优 (太小过拟合, 太大欠拟合)")
plt.tight_layout(); plt.show()

# RidgeCV 自动选 / RidgeCV
ridgecv = RidgeCV(alphas=alphas).fit(Xtr, y_tr)
print(f"RidgeCV 选出 λ = {ridgecv.alpha_:.3f}")
print(f"test R²: OLS={LinearRegression().fit(Xtr,y_tr).score(Xte,y_te):.4f}, "
      f"Ridge={ridgecv.score(Xte, y_te):.4f}")
print("California 数据特征少, Ridge 提升不大; 但高维/共线数据 Ridge 常显著胜 OLS")


<a id="7"></a>
## 7. = 高斯先验 MAP (2.10) / Ridge = Gaussian-prior MAP

**回扣 2.10**: 给参数加先验 $w_j \sim \mathcal{N}(0, \tau^2)$, 做最大后验估计：
$$\hat{\mathbf{w}}_{\text{MAP}} = \arg\max_{\mathbf{w}}\big[\underbrace{\log p(\mathbf{y}|\mathbf{w})}_{\text{高斯似然}=-\frac{1}{2\sigma^2}\|\mathbf{y}-\mathbf{X}\mathbf{w}\|^2} + \underbrace{\log p(\mathbf{w})}_{=-\frac{1}{2\tau^2}\|\mathbf{w}\|^2}\big]$$

**等价于 Ridge**, 其中 $\lambda = \sigma^2/\tau^2$。2.10 节我们已经数值验证过两者系数完全相同。

**直觉**: 正则化 = "**先验相信系数应该小**"。λ 大 = 先验更强（τ 小）= 更不信数据、更信"系数接近 0"。这给了正则化一个**概率出生证明**——不是拍脑袋加的惩罚, 而是引入先验信念。
Regularization = a prior belief that coefficients should be small. This gives it a probabilistic birth certificate.


<a id="8"></a>
## 8. 小结 / Summary

```
Ridge: min ‖y-Xw‖² + λ‖w‖²  →  ŵ = (XᵀX + λI)⁻¹Xᵀy
  +λI 让矩阵必可逆 → 治多重共线 ⭐ (系数平摊给共线特征, 稳定)
  系数平滑收缩向 0 但不为 0 → 不做特征选择 (vs Lasso)
  λ = 偏差-方差旋钮; CV/RidgeCV 选 λ
  必须先标准化; 偏置不惩罚
  = 高斯先验 MAP (2.10), λ = σ²/τ²
```

### 💡 面试速查
1. **Ridge = OLS + λ‖w‖²**, 闭式 (XᵀX+λI)⁻¹Xᵀy
2. **治共线**: +λI 让矩阵必可逆, 系数稳定平摊
3. **Ridge 不做特征选择** (系数趋0不为0); Lasso 才做
4. **λ 是偏差-方差旋钮**; 必须先标准化
5. **Ridge = 高斯先验 MAP** (2.10)

### 下一节
**4.5 Lasso (L1)**——把惩罚从 ‖w‖² 换成 ‖w‖₁, 神奇地让部分系数**精确变 0** = 自动特征选择。为什么 L1 能, L2 不能？
